# Cuaderno 4 · ¿De qué está hecha una tabla?

**Descripción y Visualización de Datos · UAI 2026 · Clase 4**

La clase pasada contaste. Hoy vamos a mirar **de qué están hechos** los datos que
contaste.

Porque una tabla no es sólo números y palabras: cada columna tiene un **tipo**, y
el tipo decide qué puedes hacer con ella. Cuando algo no funciona en R —y hoy vas
a hacer que no funcione varias veces, a propósito— nueve de cada diez veces el
problema es un tipo.

Este cuaderno tiene más celdas vacías que llenas. Ésa es la idea: correr, romper,
mirar el mensaje, arreglar.

---

**Antes de empezar: `Archivo → Guardar una copia en Drive`.**

## Parte 1 · Todo lo que escribes tiene un tipo

R no guarda solamente el valor. Guarda también **de qué clase es**. La función
que lo pregunta se llama `class()`:

In [ ]:
class(19)
class("Ñuñoa")
class(TRUE)

Salieron tres palabras. Son tres de los cuatro tipos que vas a ver todo el
semestre:

| Tipo | Qué es | Cómo se escribe |
|---|---|---|
| `numeric` | un número, con o sin decimales | `19` &nbsp;·&nbsp; `1.75` |
| `character` | texto, **siempre entre comillas** | `"Ñuñoa"` &nbsp;·&nbsp; `"19"` |
| `logical` | verdadero o falso, **sin comillas** | `TRUE` &nbsp;·&nbsp; `FALSE` |
| `integer` | un número entero, sin decimales | `19L` |

Dos cosas que suelen sorprender: `19` es `numeric` aunque no tenga decimales (R
deja el espacio por si acaso), y las comillas **no son decoración**: cambian el
tipo.

**1.** Averigua de qué clase son `19.5`, `"19"` y `FALSE`. Mira bien las comillas
antes de adivinar.

In [ ]:
# Tu código acá

## Parte 2 · El tipo decide qué puedes hacer

Con un número se puede hacer aritmética:

In [ ]:
19 + 1

Con texto no. **Corre la celda de abajo aunque sepas que va a fallar** —
justamente queremos ver cómo falla:

In [ ]:
"19" + 1

> `Error in "19" + 1 : non-numeric argument to binary operator`

Traducido: *"me pediste una operación aritmética con algo que no es un número"*.
El `19` con comillas es la palabra "19", no la cantidad 19.

**Ésta es la regla de oro del curso: lee el error entero antes de hacer
cualquier cosa.** Casi siempre dice exactamente qué pasó. Un error no es un reto,
es un mensaje.

## Parte 3 · Una columna es un vector, y un vector tiene **un solo** tipo

`c()` junta varios valores en un vector. Una columna de una tabla es exactamente
eso:

In [ ]:
horas <- c(6, 7, 8, 5)

class(horas)
mean(horas)

Perfecto: cuatro números, tipo `numeric`, promedio 6.5.

Ahora la trampa. Imagina que una de esas cuatro personas, en vez de escribir `3`,
escribió `3 horas`:

In [ ]:
horas <- c(6, 7, 8, "3 horas")

class(horas)

El vector completo se volvió `character`. R **no puede** mezclar números y texto
en un mismo vector, así que toma el camino que no pierde información: convierte
todo a texto. El `6` de arriba ahora es la palabra `"6"`.

**Un solo valor contamina la columna entera.** Y entonces esto pasa:

In [ ]:
mean(horas)

Devuelve `NA` y una advertencia (`argument is not numeric or logical`). No se
puede promediar texto.

> ¿Te acuerdas de la clase pasada? En `count(curso, horas_sueno)` aparecían `3` y
> `3 horas` como dos categorías distintas. **Ésta es la razón.** Una persona
> escribió una palabra y toda la columna dejó de ser numérica.

## Parte 4 · La misma historia, en la encuesta del curso

Cargamos sus respuestas, igual que la clase pasada:

In [ ]:
library(dplyr)

curso <- read.csv("https://raw.githubusercontent.com/naimbro/naimbro.github.io/main/materiales/2026_descripcion_visualizacion_datos/datos/encuesta_curso.csv")

nrow(curso)

`glimpse()` ya lo sabías usar, pero hoy lo vas a mirar distinto: **fíjate en la
etiqueta entre `<>` que aparece después de cada nombre de columna.** Ahí está el
tipo.

In [ ]:
glimpse(curso)

La traducción de las etiquetas:

| En `glimpse()` | `class()` lo llama | Es |
|---|---|---|
| `<int>` | `integer` | número entero |
| `<dbl>` | `numeric` | número con decimales |
| `<chr>` | `character` | texto |
| `<lgl>` | `logical` | `TRUE` / `FALSE` |

Y acá viene el hallazgo del día: **hay columnas que parecen números y son texto.**

Para revisarlas una por una necesitas sacar una columna del data frame. Eso se
hace con el signo `$`:

In [ ]:
class(curso$edad)

`integer`, como debe ser.

**2.** Ahora prueba con `curso$estatura`. Es una medida: debería ser un número.

In [ ]:
# Tu código acá

**3.** Sigue con `curso$minutos_viaje` y `curso$horas_redes`.

In [ ]:
# Tu código acá

Las tres son `character`. Y en cada una la culpa es de **un solo valor**:

| Columna | El valor que la arruinó |
|---|---|
| `estatura` | `1,60` (coma en vez de punto) |
| `minutos_viaje` | `10 min` |
| `horas_redes` | `5 horas` |

Una persona por columna. Y por eso:

In [ ]:
mean(curso$estatura)

## Parte 5 · La cura rápida, y lo que cuesta

`as.numeric()` intenta convertir texto a número. Lo que no puede convertir, lo
deja como `NA`. Y `na.rm = TRUE` le dice a `mean()`: *"ignora los `NA` y promedia
el resto"*.

In [ ]:
estatura_num <- as.numeric(curso$estatura)

mean(estatura_num, na.rm = TRUE)

**157.66 centímetros.**

Antes de seguir, para un segundo y mira ese número. Son 33 personas de primer año
de universidad. ¿Te parece que el promedio del curso es 1,58 m?

Lo que pasó son **dos fallas silenciosas**:

1. `as.numeric()` no pudo con `1,60` ni con `1,69` (la coma no es un punto) y las
   convirtió en `NA`. Con `na.rm = TRUE` el promedio se calculó sobre **31
   personas, no 33**. Nadie te avisó.
2. Peor: `1.58` y `1.76` **sí** son números válidos, así que `as.numeric()` los
   aceptó feliz. Sólo que están escritos en metros y el resto en centímetros, así
   que entraron al promedio como si esas dos personas midieran menos de dos
   centímetros.

Sin esas dos, el promedio real es **168.4 cm**. Diez centímetros de diferencia.

> `as.numeric()` + `na.rm = TRUE` **no limpia datos: los esconde**. Sirve para
> mirar rápido, no para publicar. La limpieza de verdad —decidir qué hacer con
> cada caso y dejarlo escrito— es la clase que viene, con `mutate()`.

**4.** Repite la operación con `horas_sueno`: conviértela con `as.numeric()` y
saca el promedio con `na.rm = TRUE`. ¿Cuántas horas duerme el curso? ¿Y sobre
cuántas personas quedó calculado ese promedio?

In [ ]:
# Tu código acá

## Parte 6 · Un CSV de verdad, chiquitito

Hoy en clase viste el demo del Sprint 1. Los datos de ese demo no salieron de
ningún portal: los armé **a mano**, leyendo siete estudios y anotando una fila por
cada uno. Ésa también es una fuente de datos válida, y es una que ustedes pueden
construir para su proyecto.

Acá está, tal cual:

In [ ]:
experimentos <- read.csv("https://raw.githubusercontent.com/naimbro/naimbro.github.io/main/materiales/2026_descripcion_visualizacion_datos/datos/experimentos_ia_desempeno.csv")

nrow(experimentos)
ncol(experimentos)

Siete filas, trece columnas. Un data frame diminuto y perfectamente utilizable.

In [ ]:
glimpse(experimentos)

**5.** ¿Cuántas personas participaron **en total** en los siete experimentos?
Pista: la columna se llama `n`, y la función es `sum()`.

In [ ]:
# Tu código acá

**6.** ¿Cuántos estudios hay de cada `ambito`? Ésta ya la sabes hacer desde la
clase pasada.

In [ ]:
# Tu código acá

Ahora el promedio del efecto de la IA sobre el desempeño:

In [ ]:
mean(experimentos$efecto_promedio_pct)

`NA` otra vez —pero **por una razón completamente distinta a la de la estatura**.
Acá la columna sí es numérica (`<int>` en el `glimpse`). Lo que pasa es que uno de
los siete estudios no reporta esa medida, y ese casillero quedó vacío.

R está siendo honesto: *"si hay un dato que no conozco, el promedio tampoco lo
conozco"*. Es el comportamiento correcto, no un defecto.

In [ ]:
mean(experimentos$efecto_promedio_pct, na.rm = TRUE)

**+5.8%.** Un número correcto y, al mismo tiempo, **completamente inútil**: está
promediando estudios de *trabajo* (donde la IA sube el desempeño) con estudios de
*aprendizaje* (donde lo baja). El promedio existe, pero no describe a nadie.

> Guarda esta idea para el Sprint 1: **un indicador puede estar bien calculado y
> aun así no significar nada.** La pregunta no es sólo "¿cómo lo calculo?", sino
> "¿de qué grupo estoy hablando?".

## Parte 7 · Un CSV de verdad, grande

Y ahora la escala real. La encuesta CEP: **96.122 personas** entrevistadas en
Chile entre 1994 y 2026, en un solo archivo.

La línea de abajo se demora unos segundos (son 19 MB). Córrela y espera.

In [ ]:
cep <- read.csv("https://raw.githubusercontent.com/naimbro/naimbro.github.io/main/materiales/2026_descripcion_visualizacion_datos/datos/cep_consolidada_1994_2026.csv")

nrow(cep)
ncol(cep)

In [ ]:
glimpse(cep)

Fíjate: 96 mil filas y el data frame se comporta **exactamente igual** que la
tabla de 7 filas. Las mismas funciones, escritas igual. Eso es lo que hace que
valga la pena aprender esto.

De acá en adelante exploras tú. Todo lo que sigue se responde con lo que ya
sabes: `count()`, `class()`, `mean()`, `head()`, `$`.

**7.** ¿Cuál ha sido el problema más mencionado como el principal del país en
estos 32 años? *(columna `problema_1`, de mayor a menor)*

In [ ]:
# Tu código acá

**8.** ¿Cuántas personas encuestó la CEP cada año? *(columna `anio`)*

In [ ]:
# Tu código acá

**9.** ¿Cuál es la edad promedio de las personas encuestadas? Ojo: la columna
`edad` tiene casilleros vacíos, así que vas a necesitar `na.rm = TRUE`. Y antes
de correrla, pregúntate: ¿sobre cuántas personas va a quedar calculada?

In [ ]:
# Tu código acá

**10. Tres preguntas tuyas.** Vuelve al `glimpse(cep)` de más arriba, elige las
columnas que te interesen y cuéntalas. Cruza dos si quieres —`count()` acepta dos
columnas separadas por coma—. No hay respuesta correcta: la gracia es que te dé
curiosidad algo.

Hay material de sobra: región, zona, nivel socioeconómico, religión, posición
política, aprobación del presidente, cómo ve la gente la situación económica del
país.

In [ ]:
# Tu pregunta 1

In [ ]:
# Tu pregunta 2

In [ ]:
# Tu pregunta 3

## Parte 8 · Clínica de errores (y cómo usar la IA hoy)

> **La regla de esta clase: pídele a la IA que te *explique* el mensaje, no que
> te *escriba* la línea.**
>
> No es una regla moral, es lo que muestra la evidencia que vimos hoy: cuando la
> IA entrega la respuesta hecha, el rendimiento sube mientras la usas y cae
> cuando te la quitan. Cuando explica sin resolver, el daño desaparece. Copiar la
> solución se siente igual de bien que entenderla, y ahí está el problema.
>
> Un buen prompt para hoy: *"soy principiante en R. ¿Qué significa este error?
> Explícamelo sin darme el código corregido."*

Abajo hay **tres celdas rotas**. Córrelas una por una, lee lo que sale, y
arréglalas en la celda vacía que viene después.

---

**Rota 1:**

In [ ]:
count(curso, Dominio)

In [ ]:
# Arréglala acá

**Rota 2:**

In [ ]:
class(horas_sueno)

In [ ]:
# Arréglala acá

**Rota 3:** ésta es distinta. **No da error.** Córrela y mira el resultado.

In [ ]:
mean(curso$horas_sueno)

In [ ]:
# Arréglala acá

### Las tres, resueltas

**Rota 1 — `count(curso, Dominio)`.** La columna se llama `dominio`, con
minúscula. Para R, `Dominio` y `dominio` son dos cosas distintas, igual que
`Las condes` y `las condes` eran dos comunas distintas la clase pasada. R
distingue mayúsculas **siempre**.

**Rota 2 — `class(horas_sueno)`.** Dice `object 'horas_sueno' not found`. Y tiene
razón: `horas_sueno` no existe suelta por ahí, **existe adentro de `curso`**. Hay
que ir a buscarla: `class(curso$horas_sueno)`.

**Rota 3 — `mean(curso$horas_sueno)`.** Ésta es la peligrosa: R no se cae, sólo
devuelve `NA` con una advertencia chica. La columna es `character` por culpa de
`3 horas`. Si vas apurado, copias ese `NA` a tu informe y nunca te enteras.

> **Un error te detiene. Una advertencia te deja seguir.** La segunda es la que
> hay que aprender a mirar.

## Antes de irte

**Archivo → Guardar** (Ctrl+S).

### Lo que aprendiste hoy

| Para qué | Cómo se escribe |
|---|---|
| Preguntar el tipo de algo | `class(19)` |
| Armar un vector | `c(6, 7, 8)` |
| Sacar una columna del data frame | `curso$edad` |
| Ver los tipos de todas las columnas | `glimpse(curso)` |
| Cuántas filas y cuántas columnas | `nrow(cep)` · `ncol(cep)` |
| Promediar | `mean(curso$edad)` |
| Sumar una columna | `sum(experimentos$n)` |
| Intentar convertir texto a número | `as.numeric(curso$estatura)` |
| Promediar ignorando los `NA` | `mean(x, na.rm = TRUE)` |

### Las tres ideas

1. **Un vector tiene un solo tipo.** Una persona que escribe `3 horas` convierte
   toda la columna en texto, y desde ahí no se puede promediar nada.
2. **`na.rm = TRUE` no limpia: esconde.** El promedio de estaturas daba 157.7 cm
   sobre 31 personas en vez de 168.4 sobre las 29 bien medidas, y R no se quejó
   ni una vez.
3. **La advertencia es más peligrosa que el error.** El error te obliga a mirar;
   la advertencia te deja publicar un `NA` sin darte cuenta.

### Para la próxima clase

Aprendemos a **arreglar** lo que hoy sólo diagnosticamos: `filter()`, `select()` y
`mutate()`. Con eso una columna sucia se vuelve una columna utilizable, dejando
escrito qué se decidió en cada caso.

Y se entrega el **Sprint 1**. Cuando busques tus fuentes, hazles a tus datos las
mismas tres preguntas de hoy: ¿de qué tipo es cada columna?, ¿cuántos `NA` tiene?,
¿sobre cuánta gente está calculado cada número?